In [2]:
import sys

from sklearn.model_selection import PredefinedSplit
sys.path.append("..")

import pandas as pd
import mlflow
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, f1_score
from src.temizle import Temizleyici

train = pd.read_csv("../data/processed/tr_train.csv")
test = pd.read_csv("../data/processed/tr_test.csv")
train = train[train["belirsiz"] != "e"]

X_train, y_train = train["mesaj"], train["ikili"]
X_test, y_test = test["mesaj"], test["ikili"]
cv = PredefinedSplit(train["fold"])
spam_f1 = make_scorer(f1_score, pos_label="spam")
print(X_train.shape, y_test.shape)



(651,) (116,)


In [6]:
#MLFLOW

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline


mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("tr_ikili")

model = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False)),
    ("model", LinearSVC(C=1, random_state=55)),
])

with mlflow.start_run(run_name="svm_char_wb"):
    skor = cross_val_score(model, X_train, y_train, cv=cv, scoring=spam_f1)
    mlflow.log_params({"model": "LinearSVC", "feature": "char_wb 3-5", "C": 1})
    mlflow.log_metrics({"cv_f1": skor.mean(), "cv_f1_std": skor.std()})
    print(skor.mean().round(3))

0.971


In [7]:
def parametreler(model):
    p = model.get_params()
    return {
        "model": type(p["model"]).__name__,
        "analyzer": p["vek__analyzer"],
        "ngram_range": str(p["vek__ngram_range"]),
        "C": p["model__C"],
    }

print(parametreler(model))

{'model': 'LinearSVC', 'analyzer': 'char_wb', 'ngram_range': '(3, 5)', 'C': 1}


In [9]:
for md in [1, 2]:
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False, min_df=md)
    v.fit(Temizleyici().transform(X_train))
    print(f"min_df={md}: {len(v.vocabulary_)} parça")

min_df=1: 25076 parça
min_df=2: 12616 parça


In [10]:
def parametreler(model):
    p = model.get_params()
    sonuc = {
        "model": type(p["model"]).__name__,
        "analyzer": p["vek__analyzer"],
        "ngram_range": str(p["vek__ngram_range"]),
        "min_df": p["vek__min_df"],
    }
    for ad, deger in p.items():
        if ad.startswith("model__") and isinstance(deger, (int, float, str, tuple)):
            sonuc[ad] = str(deger)
    return sonuc

In [11]:
from sklearn.neural_network import MLPClassifier

mlp = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False, min_df=2)),
    ("model", MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=300, random_state=42)),
])

with mlflow.start_run(run_name="mlp_char_wb"):
    skor = cross_val_score(mlp, X_train, y_train, cv=cv, scoring=spam_f1)
    mlflow.log_params(parametreler(mlp))
    mlflow.log_metrics({"cv_f1": skor.mean(), "cv_f1_std": skor.std()})
    print(skor.mean().round(3), skor.round(3))

0.974 [0.953 0.968 0.992 0.975 0.983]


In [12]:
from sklearn.model_selection import cross_validate

for ad, m in [("SVM", model), ("MLP", mlp)]:
    s = cross_validate(m, X_train, y_train, cv=cv, scoring=spam_f1, return_train_score=True)
    print(f"{ad}: train {s['train_score'].mean():.3f}  cv {s['test_score'].mean():.3f}  fark {s['train_score'].mean() - s['test_score'].mean():.3f}")

SVM: train 1.000  cv 0.971  fark 0.029
MLP: train 1.000  cv 0.974  fark 0.026
